# Spaceship Titanic

Predict which passengers were transported to an alternate dimension.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

## 1. Load Data

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

train.shape, test.shape

In [ ]:
train.head()

In [ ]:
train.info()

## 2. EDA

In [ ]:
numerical_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
sns.pairplot(train, vars=numerical_cols, hue="Transported", diag_kind="kde", plot_kws={"alpha": 0.3, "s": 10}, corner=True, height=2.5)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(train[numerical_cols + ["Transported"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

In [ ]:
categorical_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]

fig, axes = plt.subplots(1, len(categorical_cols), figsize=(16, 4))
for ax, col in zip(axes, categorical_cols):
    train.groupby(col)["Transported"].mean().plot.bar(ax=ax, rot=0)
    ax.set_ylabel("Transport rate")
    ax.set_title(col)
    ax.set_ylim(0, 1)
    ax.axhline(train["Transported"].mean(), color="red", linestyle="--", label="overall")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for col in categorical_cols:
    print(f"--- {col} ---")
    print(train[col].value_counts(dropna=False))
    print()

In [ ]:
train["Cabin"].head(10)

In [ ]:
train["Transported"].value_counts().plot.bar(rot=0)
plt.title("Target balance")
plt.show()

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"{'Column':<15} {'Missing':>7} {'%':>6}")
print("-" * 30)
for col, count in missing.items():
    print(f"{col:<15} {count:>7} {count / len(train) * 100:>5.1f}%")

In [ ]:
train.isnull().sum(axis=1).value_counts().sort_index()

## 3. Preprocessing & Feature Engineering

In [ ]:
bool_cols = ["Transported", "CryoSleep", "VIP"]
for col in bool_cols:
    train[col] = train[col].map({True: 1, False: 0, "True": 1, "False": 0})

train[bool_cols].head()

In [ ]:
train["PassengerId"] = train["PassengerId"].str.split("_").str[0].astype(int)

train["PassengerId"].head()

In [ ]:
train = pd.get_dummies(train, columns=["HomePlanet", "Destination"], dtype=int)

train.head()

In [ ]:
cabin_split = train["Cabin"].str.split("/", expand=True)
train["CabinDeck"] = cabin_split[0]
train["CabinNum"] = cabin_split[1].astype(float)
train["CabinSide"] = cabin_split[2].map({"P": 0, "S": 1})
train.drop(columns=["Cabin"], inplace=True)

train = pd.get_dummies(train, columns=["CabinDeck"], dtype=int)

train[["CabinNum", "CabinSide"] + [c for c in train.columns if c.startswith("CabinDeck_")]].head()

In [ ]:
train["LastName"] = train["Name"].str.split().str[-1]
train.drop(columns=["Name"], inplace=True)

train["LastName"].head()

In [ ]:
train["FamilySize"] = train.groupby("LastName")["LastName"].transform("count")
train.drop(columns=["LastName"], inplace=True)

train["FamilySize"].value_counts().sort_index()

In [ ]:
from sklearn.impute import KNNImputer

spending_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# CryoSleep passengers spent 0
for col in spending_cols:
    train.loc[train["CryoSleep"] == 1, col] = train.loc[train["CryoSleep"] == 1, col].fillna(0)

# KNN imputation for remaining NaNs (fit on features only, not target)
feature_cols = [c for c in train.columns if c != "Transported"]
imputer = KNNImputer(n_neighbors=5)
train[feature_cols] = imputer.fit_transform(train[feature_cols])

print("Remaining NaNs:", train.isnull().sum().sum())

In [ ]:
spending_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
train["TotalSpending"] = train[spending_cols].sum(axis=1)
train["NoSpending"] = (train["TotalSpending"] == 0).astype(int)
train["AgeGroup"] = pd.cut(train["Age"], bins=[0, 12, 18, 40, 60, 100], labels=[0, 1, 2, 3, 4]).astype(float)
for col in spending_cols:
    train[f"{col}_ratio"] = train[col] / train["TotalSpending"].replace(0, 1)

# Log transform spending features
for col in spending_cols + ["TotalSpending"]:
    train[f"Log_{col}"] = np.log1p(train[col])

train[["TotalSpending", "Log_TotalSpending", "NoSpending"]].head()

## 4. Baseline Model

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

X = train.drop(columns=["Transported"])
y = train["Transported"]

rf = RandomForestClassifier(random_state=42)
scores = cross_val_score(rf, X, y, cv=10, scoring="accuracy")

print(f"Fold scores: {scores}")
print(f"Mean accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 5. Improved Model

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

models = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, verbosity=0),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=10, scoring="accuracy")
    print(f"{name:15s}  {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def xgb_objective(trial):
    clf = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 2000),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        learning_rate=trial.suggest_float("lr", 0.005, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample", 0.3, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        random_state=42, verbosity=0, n_jobs=-1, device="cuda",
    )
    return cross_val_score(clf, X, y, cv=5, scoring="accuracy").mean()

xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_objective, n_trials=200, show_progress_bar=True)

print(f"\nBest XGBoost accuracy: {xgb_study.best_value:.4f}")
print(f"Best params: {xgb_study.best_params}")

In [ ]:
from sklearn.ensemble import VotingClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

best_xgb = XGBClassifier(
    n_estimators=139, max_depth=9, learning_rate=0.0108,
    subsample=0.511, colsample_bytree=0.451, min_child_weight=20,
    gamma=2.085, reg_alpha=6.311, reg_lambda=6.231,
    random_state=42, verbosity=0, n_jobs=-1,
)
best_lgb = LGBMClassifier(
    n_estimators=133, max_depth=9, learning_rate=0.0102,
    subsample=0.797, colsample_bytree=0.734, min_child_weight=1.65,
    num_leaves=14, reg_alpha=1.13e-6, reg_lambda=3.71e-7,
    random_state=42, verbose=-1, n_jobs=-1,
)
best_et = ExtraTreesClassifier(
    n_estimators=260, max_depth=13, min_samples_split=16, min_samples_leaf=8,
    random_state=42, n_jobs=-1,
)

stacker = StackingClassifier(
    estimators=[("xgb", best_xgb), ("lgb", best_lgb), ("et", best_et)],
    final_estimator=LogisticRegression(),
    cv=5,
    passthrough=False,
    n_jobs=-1,
)

scores = cross_val_score(stacker, X, y, cv=10, scoring="accuracy")
print(f"Stacking (XGB+LGBM+ET): {scores.mean():.4f} (+/- {scores.std():.4f})")

## 6. Submission

In [ ]:
test_raw = pd.read_csv("data/test.csv")
test_ids = test_raw["PassengerId"].copy()

# Same preprocessing as train
test = test_raw.copy()
for col in ["CryoSleep", "VIP"]:
    test[col] = test[col].map({True: 1, False: 0, "True": 1, "False": 0})
test["PassengerId"] = test["PassengerId"].str.split("_").str[0].astype(int)
test = pd.get_dummies(test, columns=["HomePlanet", "Destination"], dtype=int)

cabin_split = test["Cabin"].str.split("/", expand=True)
test["CabinDeck"] = cabin_split[0]
test["CabinNum"] = cabin_split[1].astype(float)
test["CabinSide"] = cabin_split[2].map({"P": 0, "S": 1})
test.drop(columns=["Cabin"], inplace=True)
test = pd.get_dummies(test, columns=["CabinDeck"], dtype=int)

test["LastName"] = test["Name"].str.split().str[-1]
test.drop(columns=["Name"], inplace=True)
test["FamilySize"] = test.groupby("LastName")["LastName"].transform("count")
test.drop(columns=["LastName"], inplace=True)

# CryoSleep passengers spent 0
for col in spending_cols:
    test.loc[test["CryoSleep"] == 1, col] = test.loc[test["CryoSleep"] == 1, col].fillna(0)

# KNN imputation (use imputer fitted on train feature_cols)
# Align test columns to match imputer before transform
for col in feature_cols:
    if col not in test.columns:
        test[col] = 0
test[feature_cols] = imputer.transform(test[feature_cols])

# Engineered features
test["TotalSpending"] = test[spending_cols].sum(axis=1)
test["NoSpending"] = (test["TotalSpending"] == 0).astype(int)
test["AgeGroup"] = pd.cut(test["Age"], bins=[0, 12, 18, 40, 60, 100], labels=[0, 1, 2, 3, 4]).astype(float)
for col in spending_cols:
    test[f"{col}_ratio"] = test[col] / test["TotalSpending"].replace(0, 1)

# Log transform spending features
for col in spending_cols + ["TotalSpending"]:
    test[f"Log_{col}"] = np.log1p(test[col])

# Align columns with train
for col in X.columns:
    if col not in test.columns:
        test[col] = 0
test = test[X.columns]

# Train on full data and predict
stacker.fit(X, y)
preds = stacker.predict(test)

submission = pd.DataFrame({"PassengerId": test_ids, "Transported": preds.astype(bool)})
submission.to_csv("data/submission.csv", index=False)
print(submission.head())
print(f"Submission shape: {submission.shape}")